In [ ]:
# 这个code为了测试as插值方法对于多少个点要运行多久，有均匀grid插值，也有随机生成固定数量的点插值（后者没有验证）
##########################################################################################
import pyomo.environ as pyo
from pyomo.environ import *
from pyomo.contrib.piecewise import PiecewiseLinearFunction
from pyomo.opt import SolverFactory
from pyomo.core.base import TransformationFactory
from pyomo.opt import SolverStatus, TerminationCondition
from dataclasses import dataclass, field
from typing import Callable, List, Dict, Tuple, Optional
import numpy as np
import math
import matplotlib.pyplot as plt
import bisect
import itertools as it
from tqdm import tqdm
import csv
from pyomo.solvers.plugins.solvers.gurobi_persistent import GurobiPersistent
from typing import List, Tuple
import pandas as pd
import matplotlib.pyplot as plt
import time
from pyomo.core.base import TransformationFactory
from pyomo.contrib.piecewise import PiecewiseLinearFunction as PLF


USE_DLOG = True  # True=更快(二进制 O(logN))，False=CC(二进制 O(N))

def apply_pw_transform(m):
    tx = ("contrib.piecewise.disaggregated_logarithmic"
          if USE_DLOG else
          "contrib.piecewise.convex_combination")
    TransformationFactory(tx).apply_to(m)

def assert_no_plf_left(m, where=""):
    leftovers = [c.name for c in m.component_objects(PLF, active=True)]
    if leftovers:
        raise RuntimeError(f"[{where}] 还有未线性化的 PiecewiseLinearFunction: {leftovers}")


# ----------------------- 工具函数：统一设置 Gurobi 参数 -----------------------
def apply_params(opt: GurobiPersistent):
    opt.set_gurobi_param('MIPGap',         1e-2)
    opt.set_gurobi_param('FeasibilityTol', 1e-6)
    opt.set_gurobi_param('IntFeasTol',     1e-6)
    opt.set_gurobi_param('OptimalityTol',  1e-6)
    opt.set_gurobi_param('NumericFocus',   1)
    opt.set_gurobi_param('Presolve',       2)
    opt.set_gurobi_param('NonConvex',      2)
    #opt.set_gurobi_param('TimeLimit',      60)   # ★ 每次最多15秒，可按需 10/20/30
    # 也可试：opt.set_gurobi_param('MIPFocus', 1)  # 1=找可行优质解


# ----------------------- 求场景真值 v(y) -----------------------
def evaluate_Q_at(model, first_stg_vars, first_stg_vals, solver):
    del_components(model)
    for u, v in zip(first_stg_vars, first_stg_vals):
        u.fix(pyo.value(v))
    model.obj = pyo.Objective(expr=model.obj_expr, sense=pyo.minimize)

    try:
        # 若是 persistent，先绑定，再不传 model 求解；否则按 file-based 调用
        if hasattr(solver, "set_instance"):
            solver.set_instance(model)
            results = solver.solve(tee=False)
        else:
            results = solver.solve(model, tee=False)

        status_ok = (results.solver.status == SolverStatus.ok)
        term_ok   = (results.solver.termination_condition == TerminationCondition.optimal)
        if not (status_ok and term_ok):
            raise RuntimeError(
                f"Scenario evaluate at y={first_stg_vals} not optimal: "
                f"status={results.solver.status}, term={results.solver.termination_condition}"
            )
        return pyo.value(model.obj_expr)
    finally:
        if hasattr(model, 'obj'):
            model.del_component('obj')
        for u in first_stg_vars:
            if u.fixed:
                u.unfix()

# ----------------------- 打印辅助 -----------------------
def _fmt_item(x, prec=6):
    if isinstance(x, (tuple, list)):
        return "(" + ", ".join(f"{float(v):.{prec}f}" for v in x) + ")"
    try:
        return f"{float(x):.{prec}f}"
    except Exception:
        return str(x)

# ----------------------- warm start -----------------------
def dump_solution(m):
    return {
        'Kp': pyo.value(m.Kp),
        'Ki': pyo.value(m.Ki),
        'Kd': pyo.value(m.Kd),
        'x': {t: pyo.value(m.x[t]) for t in m.T},
        'u': {t: pyo.value(m.u[t]) for t in m.T},
        'e': {t: pyo.value(m.e[t]) for t in m.T},
        'I': {t: pyo.value(m.I[t]) for t in m.T},
    }



def apply_warm_values(m, sol):
    """极简 warm start：写 value 与 start，并做取整+越界裁剪。
       仅支持 dump_solution(m) 的结构：标量 Kp/Ki/Kd 与时序 x/u/e/I。
    """
    if not sol:
        return

    def _write(v, val):
        if val is None:
            return
        x = float(val)

        # 越界裁剪
        if v.has_lb() and x < v.lb: x = float(v.lb)
        if v.has_ub() and x > v.ub: x = float(v.ub)

        # 整数/二元取整
        if v.is_integer() or v.is_binary():
            x = int(round(x))
            if v.is_binary():
                x = 1 if x >= 1 else 0
            x = float(x)

        # 写回
        v.value = x       # 可行性初始化
        try:
            v.set_start(x)  # file-based 导出为 MIP start
        except Exception:
            v.start = x

    # 标量
    for name in ("Kp", "Ki", "Kd"):
        if hasattr(m, name) and name in sol:
            _write(getattr(m, name), sol.get(name))

    # 时序
    for name in ("x", "u", "e", "I"):
        if hasattr(m, name) and name in sol and isinstance(sol[name], dict):
            var = getattr(m, name)
            for t, val in sol[name].items():
                try:
                    _write(var[int(t)], val)  # 你的索引通常是 0..T
                except Exception:
                    try:
                        _write(var[t], val)
                    except Exception:
                        pass



def push_start_to_gurobi(opt, m):
    for v in m.component_data_objects(pyo.Var, active=True):
        val = v.value
        if val is None:
            continue
        if v.is_binary() or v.is_integer():
            if abs(val - round(val)) < 1e-6:
                val = int(round(val))
        if v.has_lb() and val < v.lb: val = v.lb
        if v.has_ub() and val > v.ub: val = v.ub
        opt.set_var_attr(v, "Start", float(val))

# ----------------------- 打印节点表 -----------------------
def print_nodes_row(existing_nodes, new_node,
                    existing_values=None, new_value=None,
                    prec_node=2, prec_value=6, pad=2, label_new="(new node)",
                    highlight_min=True):
    headers = [f"node{i+1}" for i in range(len(existing_nodes))] + [f"node{len(existing_nodes)+1} {label_new}"]
    node_strs = [_fmt_item(n, prec_node) for n in existing_nodes] + [_fmt_item(new_node, prec_node)]

    have_vals = existing_values is not None or new_value is not None
    if existing_values is None:
        existing_values = [None] * len(existing_nodes)

    val_strs = []
    if have_vals:
        for v in existing_values:
            val_strs.append(_fmt_item(v, prec_value) if v is not None else "")
        val_strs.append(_fmt_item(new_value, prec_value) if new_value is not None else "")

    min_val = None
    if have_vals and highlight_min:
        try:
            nums = [float(v) for v in existing_values if v is not None]
            if new_value is not None:
                nums.append(float(new_value))
            if nums:
                min_val = min(nums)
        except Exception:
            pass

    cols = max(len(headers), len(node_strs))
    widths = []
    for j in range(cols):
        pieces = []
        if j < len(headers):   pieces.append(headers[j])
        if j < len(node_strs): pieces.append(node_strs[j])
        if have_vals and j < len(val_strs): pieces.append(val_strs[j])
        w = max(len(s) for s in pieces) + pad*2
        widths.append(w)

    def _center(s, w): return s.center(w)
    def _right(s, w):  return s.rjust(w)

    header_line = "".join(_center(h, widths[i]) for i, h in enumerate(headers))
    sep_line = "".join("-" * widths[i] for i in range(len(headers)))
    print(header_line)
    print(sep_line)

    node_line = "".join(_right(s, widths[i]) for i, s in enumerate(node_strs))
    print(node_line)

    if have_vals:
        val_line_parts = []
        for i, s in enumerate(val_strs):
            if s and min_val is not None and abs(float(s) - min_val) < 1e-12:
                colored = f"\033[31m{s}\033[0m"
                val_line_parts.append(_right(colored, widths[i] + 9))
            else:
                val_line_parts.append(_right(s, widths[i]))
        print("".join(val_line_parts))

# ----------------------- 组件清理 -----------------------
def del_components(model):
    for comp in ['obj', 'As', 'pw', 'pw_fun', 'pw_As', 'pw_link']:
        if hasattr(model, comp):
            model.del_component(comp)

# ----------------------- 角点生成 -----------------------
def corners_from_bounds(firt_stg_vars):
    bounds = []
    for y in firt_stg_vars:
        lb, ub = y.lb, y.ub
        if lb is None or ub is None:
            raise ValueError(f"{y.name} 缺少上下界，无法生成角点")
        bounds.append((float(lb), float(ub)))
    return list(it.product(*[(lb, ub) for (lb, ub) in bounds]))

# ----------------------- n 维分片 -----------------------
def add_nd_piecewise(model, firt_stg_vars, points, values,
                     name="pw", relation="==", round_ndigits=12):
    if len(points) == 0:
        raise ValueError("points 不能为空")
    N = len(firt_stg_vars)
    for pt in points:
        if len(pt) != N:
            raise ValueError(f"points 中出现与 x_vars 维度不一致的点: {pt}")
    del_components(model)

    def keyize(coords):
        return tuple(round(float(c), round_ndigits) for c in coords)
    norm_points = [keyize(pt) for pt in points]

    if isinstance(values, dict):
        table = {keyize(k): float(v) for k, v in values.items()}
        miss = [pt for pt in norm_points if pt not in table]
        if miss:
            raise KeyError(f"values 缺少这些点的取值: {miss[:5]}{' ...' if len(miss)>5 else ''}")
    else:
        if len(values) != len(points):
            raise ValueError("values 长度应与 points 数量一致（或传 dict）")
        table = {pt: float(v) for pt, v in zip(norm_points, values)}

    def _f_from_table(*coords):
        return table[keyize(coords)]

    pw = PiecewiseLinearFunction(points=norm_points, function=_f_from_table, name=f"{name}_fun")
    model.add_component(pw.name, pw)
    pw_expr = pw(*firt_stg_vars)

    As = Var(name=f"{name}_As")
    model.add_component(As.name, As)
    if relation == "==":
        link = Constraint(expr=As == pw_expr)
    elif relation == ">=":
        link = Constraint(expr=As >= pw_expr)
    elif relation == "<=":
        link = Constraint(expr=As <= pw_expr)
    else:
        raise ValueError("relation 只能是 '==', '>=', '<='")
    model.add_component(f"{name}_link", link)

    # ← 这里不做 Transformation；在外面统一调用 apply_pw_transform(model)
    return As, pw


# ----------------------- 克隆模板 -----------------------
def clone_and_get_vars(m_old, first_stage_vars):
    m_new = m_old.clone()
    first_stage_vars_new = []
    for v in first_stage_vars:
        v_new = m_new.find_component(v.name)
        if v_new is None:
            raise KeyError(f"在新模型里找不到变量 '{v.name}'")
        first_stage_vars_new.append(v_new)
    return m_new, first_stage_vars_new

# ----------------------- 去重 -----------------------
def unique_points(points, atol=1e-9):
    out = []
    for p in points:
        if not any(all(abs(a-b) <= atol for a,b in zip(p, q)) for q in out):
            out.append(p)
    return out

# ----------------------- CSV 读取 & 模型构建 -----------------------
def load_scenarios_from_csv(csv_path: str, T: Optional[int] = None,
                            sp0: float = 0.0, sp1: float = 0.5,
                            ku_col: str = "tau_us", tau_col: str = "tau_xs",
                            disturb_prefix: str = "disturbance_",
                            setpoint_change_col: str = "setpoint_change") -> Tuple[List[Dict], int]:
    scens: List[Dict] = []
    with open(csv_path, "r", newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        fieldnames = reader.fieldnames or []
        if T is None:
            max_idx = -1
            for name in fieldnames:
                if name.startswith(disturb_prefix):
                    try:
                        k = int(name[len(disturb_prefix):])
                        max_idx = max(max_idx, k)
                    except:
                        pass
            if max_idx < 0:
                raise ValueError(f"未找到 '{disturb_prefix}k' 格式的扰动列")
            T = max_idx

    with open(csv_path, "r", newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            if ku_col not in row or tau_col not in row:
                raise KeyError(f"CSV 缺少必要列: '{ku_col}' 或 '{tau_col}'")
            Ku  = float(row[ku_col]); tau = float(row[tau_col])

            d = []
            for t in range(T+1):
                col = f"{disturb_prefix}{t}"
                if col not in row:
                    raise KeyError(f"CSV 缺少扰动列: {col}")
                d.append(float(row[col]))

            sp = [sp1]*(T+1)
            if setpoint_change_col in row and row[setpoint_change_col] != "":
                try:
                    t_star = int(float(row[setpoint_change_col]))
                    for t in range(T+1):
                        sp[t] = sp0 if t < t_star else sp1
                except:
                    pass

            scens.append({"Ku": Ku, "tau": tau, "d": d, "sp": sp})
    return scens, T

def build_pid_model(T=10, h=0.2, scen=None, weights=(1.0, 0.01),
                    bounds=None, use_cvar=False, alpha=0.95):
    assert scen is not None, "请提供一个场景字典"
    Ku, tau, d, sp = scen["Ku"], scen["tau"], scen["d"], scen["sp"]
    assert len(d) == T+1 and len(sp) == T+1

    if bounds is None:
        bounds = {}
    bx = bounds.get("x",  (-20, 20))
    bu = bounds.get("u",  (None, None))
    bKp= bounds.get("Kp", (0, 100))
    bKi= bounds.get("Ki", (0, 100))
    bKd= bounds.get("Kd", (0, 100))
    be = bounds.get("e",  (-100, 100))
    bI = bounds.get("I",  (-200, 200))

    m = pyo.ConcreteModel()
    m.T  = pyo.RangeSet(0, T)
    m.Tm = pyo.RangeSet(1, T)

    m.Kp = pyo.Var(bounds=bKp)
    m.Ki = pyo.Var(bounds=bKi)
    m.Kd = pyo.Var(bounds=bKd)

    m.x = pyo.Var(m.T, bounds=bx)
    m.u = pyo.Var(m.T, bounds=bu)
    m.e = pyo.Var(m.T, bounds=be)
    m.I = pyo.Var(m.T, bounds=bI)

    def _err_rule(m, t): return m.e[t] == sp[t] - m.x[t]
    m.err_def = pyo.Constraint(m.T, rule=_err_rule)

    def _I_dyn(m, t): return m.I[t] == m.I[t-1] + h*m.e[t]
    m.I_dyn = pyo.Constraint(m.Tm, rule=_I_dyn)

    def _x_dyn(m, t):
        return m.x[t] == m.x[t-1] + (h/tau)*(-m.x[t] + Ku*m.u[t] + d[t])
    m.x_dyn = pyo.Constraint(m.Tm, rule=_x_dyn)

    def _pid_rule(m, t):
        if t == 0:
            return m.u[t] == m.Kp*m.e[t] + m.Ki*m.I[t]
        return m.u[t] == m.Kp*m.e[t] + m.Ki*m.I[t] + m.Kd*(m.e[t]-m.e[t-1])/h
    m.pid = pyo.Constraint(m.T, rule=_pid_rule)

    m.x0 = pyo.Constraint(expr=m.x[0] == 0)
    m.I0 = pyo.Constraint(expr=m.I[0] == 0)

    w_e, w_u = weights
    m.cost = pyo.Expression(expr=sum(h*(w_e*m.e[t]**2 + w_u*m.u[t]**2) for t in m.T))
    m.obj_expr = pyo.Expression(expr=m.cost)
    return m, [m.Kp, m.Ki, m.Kd]

def build_models_from_csv(csv_path: str, h: float = 0.2,
                          weights=(1.0, 0.01), bounds=None,
                          sp0: float = 0.0, sp1: float = 0.5,
                          ku_col: str = "tau_us", tau_col: str = "tau_xs",
                          disturb_prefix: str = "disturbance_",
                          setpoint_change_col: str = "setpoint_change",
                          max_scenarios=None, skip=0):
    scens, T = load_scenarios_from_csv(
        csv_path=csv_path, T=None, sp0=sp0, sp1=sp1,
        ku_col=ku_col, tau_col=tau_col,
        disturb_prefix=disturb_prefix, setpoint_change_col=setpoint_change_col,
    )
    if skip or max_scenarios:
        scens = scens[skip: (skip + max_scenarios) if max_scenarios else None]

    model_list, first_stg_vars_list = [], []
    for scen in scens:
        m, yvars = build_pid_model(T=T, h=h, scen=scen, weights=weights, bounds=bounds)
        model_list.append(m)
        first_stg_vars_list.append(yvars)

    m_tmpl_list = [model_list[0], first_stg_vars_list[0]]
    return model_list, first_stg_vars_list, m_tmpl_list, T
#---------------------------------------------------------------------------------------------
#---------------------------------------------------------------------------------------------
#---------------------------------------------------------------------------------------------
#---------------------------------------------------------------------------------------------
#---------------------------------------------------------------------------------------------
# ----------------------- 主算法：underestimator ----------------------------------------------
#---------------------------------------------------------------------------------------------
#---------------------------------------------------------------------------------------------
#---------------------------------------------------------------------------------------------
#---------------------------------------------------------------------------------------------
#---------------------------------------------------------------------------------------------
def nc_underest(model_list, first_stg_vars_list, m_tmpl_list, target_nodes,
                solver, tolerance=1e-8, probs=None, persistent_solvers=None):
    assert persistent_solvers is not None and len(persistent_solvers) == len(model_list), \
        "需要传入与 model_list 同长度的 persistent_solvers 列表"

    N = len(model_list)
    if probs is None:
        probs = [1.0]*N

    as_nodes_list = [[] for _ in range(N)]
    ms_list = [None] * N
    new_nodes_list = [None] * N
    LB_list = []
    add_node_history = []

    first_stg_nodes = corners_from_bounds(first_stg_vars_list[0])
    for i in range(N):
        as_nodes_list[i].extend(
            evaluate_Q_at(model_list[i], first_stg_vars_list[i], node, solver)
            for node in first_stg_nodes
        )
    print('corner nodes are ', first_stg_nodes)
    print('as_nodes_list are ', as_nodes_list)

    if target_nodes <= len(first_stg_nodes):
        print('target_nodes number should be larger than ', len(first_stg_nodes))
        return

    print('Start from ', len(first_stg_nodes), ' corner nodes')
    print('The goal is to get ', target_nodes, ' nodes')
    k_list = []

    # 复用一个 sum_opt（每轮重绑和设参）
    sum_opt = GurobiPersistent()

    for k in tqdm(range(len(first_stg_nodes)+1, target_nodes+1), desc="Adding nodes"):
        print('##################################################')
        print('Start adding node ', k)
        k_list.append(k)

        for i in range(N):
            print('\nSolving scenario ', i)
            del_components(model_list[i])
            As, _ = add_nd_piecewise(
                model_list[i],
                first_stg_vars_list[i],
                first_stg_nodes,
                as_nodes_list[i],
                name=f"pw_scen_{k}_{i}",
                relation="=="
            )

            # 2) 目标函数：min obj_expr - As_i
            model_list[i].obj = Objective(expr=model_list[i].obj_expr - As, sense=minimize)

            # 先线性化，再绑定 persistent
            apply_pw_transform(model_list[i])
            assert_no_plf_left(model_list[i], where=f"scenario-{i}")
            apply_warm_values(model_list[i], warm_solutions[i])

            '''
            #apply_warm_values(model_list[i], warm_solutions[i])
            opt = persistent_solvers[i]
            opt.set_instance(model_list[i])      # 现在才绑定
            apply_params(opt)
            push_start_to_gurobi(opt, model_list[i])
            '''

            start = time.time()
            #results = opt.solve(tee=True)
            results = solver.solve(model_list[i], tee=True)
            end = time.time()

            # (可选) 统计 Start 覆盖率
            n_total = n_started = 0
            for v in model_list[i].component_data_objects(pyo.Var, active=True):
                n_total += 1
                if v.value is not None:
                    n_started += 1
            print(f"[warm-start] wrote Start for {n_started}/{n_total} vars")

            print('**************************************************')
            print('**************************************************')
            print(f"iteration {k}, scenario {i}, 计算Q与As最大差值（ms）用时: {end - start:.4f} 秒")
            print('**************************************************')
            print('**************************************************')


            if (results.solver.status != SolverStatus.ok) or \
               (results.solver.termination_condition != TerminationCondition.optimal):
                print("⚠ There may be problems with the solution")

            ms_list[i] = value(model_list[i].obj)
            new_nodes_list[i] = tuple(value(v) for v in first_stg_vars_list[i])
            print('new node is ', new_nodes_list[i])
            print('ms is ', ms_list[i])

            warm_solutions[i] = dump_solution(model_list[i])

        arr = np.array(as_nodes_list, dtype=float, ndmin=2)
        assum_nodes = arr.sum(axis=0)

        print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
        print(first_stg_nodes)
        print(assum_nodes)

        model_sum, model_sum_first_stg_vars = clone_and_get_vars(m_tmpl_list[0], m_tmpl_list[1])
        del_components(model_sum)
        As_sum, pw_sum = add_nd_piecewise(
            model_sum, model_sum_first_stg_vars, first_stg_nodes, assum_nodes,
            name="pw", relation="=="
        )
        model_sum.obj = Objective(expr=As_sum, sense=minimize)

        apply_pw_transform(model_sum)
        assert_no_plf_left(model_sum, where="sum")

        '''
        # sum_opt：先绑定再设参
        sum_opt.set_instance(model_sum)
        apply_params(sum_opt)
        '''
        
        start = time.time()
        #results = sum_opt.solve(tee=True)
        results = solver.solve(model_sum, tee=True)
        end = time.time()

        print('**************************************************')
        print('**************************************************')
        print(f"iteration {k}, 计算这轮As最小值用时: {end - start:.4f} 秒")
        print('**************************************************')
        print('**************************************************')
        if not ((results.solver.status == SolverStatus.ok) and
                (results.solver.termination_condition == TerminationCondition.optimal)):
            print("Sum model doesn't get solved normally")

        As_min = results.problem.lower_bound
        print(f'As_min at possible new node is {As_min}')
        node_star = tuple(value(v) for v in model_sum_first_stg_vars)

        if (node_star is None) or (node_star in first_stg_nodes):
            avg = []
            for j in range(len(first_stg_nodes[0])):
                comp_vals = [node[j] for node in first_stg_nodes]
                avg.append(sum(comp_vals) / len(comp_vals))
            node_star = tuple(avg)
            As_min = value(pw_sum(*node_star))

        q_node_star = 0.0
        for i in range(N):
            q_node_star += evaluate_Q_at(model_list[i], first_stg_vars_list[i], node_star, solver)
        print(f'Real value at possible new node is {q_node_star}')
        errors_node_star = abs(As_min - q_node_star)
        print(f'error at possible new node is {errors_node_star}')

        sum_ms = sum(ms_i for ms_i in ms_list)

        print('Sum *****************************************')
        print('node_star is ', node_star)
        print('error at node_star is ', errors_node_star)
        print('ms_list and sum_ms is ', ms_list, sum_ms)

        if errors_node_star > abs(sum_ms):
            new_node = node_star
            new_node_value = q_node_star
            print('new node choosen from error')
        else:
            min_index = int(np.argmin(ms_list))
            new_node = new_nodes_list[min_index]
            new_node_value = 0
            for i in range(N):
                new_node_value += evaluate_Q_at(model_list[i], first_stg_vars_list[i], new_node, solver)
            print('new node choosen from ms')

        LB_list.append(As_min + sum_ms)
        add_node_history.append(new_node)
        print_nodes_row(first_stg_nodes, new_node,
                        existing_values=assum_nodes,
                        new_value=new_node_value,
                        prec_node=1,
                        prec_value=6)
        print('new node is', new_node)
        print('Current As_min is', LB_list[-1])
        print('*****************************************\n')

        print('current nodes are ', first_stg_nodes)
        print('as_nodes_list are ', as_nodes_list)
        print('new_node is', new_node)
        first_stg_nodes.append(new_node)
        for i in range(N):
            as_val = evaluate_Q_at(model_list[i], first_stg_vars_list[i], new_node, solver)
            as_nodes_list[i].append(as_val)

        arr = np.array(as_nodes_list, dtype=float, ndmin=2)
        assum_nodes = arr.sum(axis=0)

        idx = int(np.argmin(assum_nodes))
        val = assum_nodes[idx]
        print(f"\033[31mcurrent node is {idx+1} node, value is {val:.6f}\033[0m")
        print(f"node is {first_stg_nodes[idx]}")

    arr = np.array(as_nodes_list, dtype=float, ndmin=2)
    assum_nodes = arr.sum(axis=0)

    model_sum, model_sum_first_stg_vars = clone_and_get_vars(m_tmpl_list[0], m_tmpl_list[1])
    del_components(model_sum)
    As_sum, pw_sum = add_nd_piecewise(
        model_sum, model_sum_first_stg_vars, first_stg_nodes, assum_nodes,
        name="pw", relation="=="
    )
    model_sum.obj = Objective(expr=As_sum, sense=minimize)

    # ✅ 先线性化，再绑定 persistent
    apply_pw_transform(model_sum)                 # DLOG/CC 由 USE_DLOG 控制
    assert_no_plf_left(model_sum, where="sum")    # 若还有 pw_fun，直接在这里报错
    '''
    sum_opt.set_instance(model_sum)               # 现在才绑定
    apply_params(sum_opt)
    '''

    #results = sum_opt.solve(tee=True)
    results = solver.solve(model_sum, tee=True)
    

    if not ((results.solver.status == SolverStatus.ok) and
            (results.solver.termination_condition == TerminationCondition.optimal)):
        print("Sum model doesn't get solved normally")

    output_lb = results.problem.lower_bound + sum(ms_list)
    print('lower bound is ', output_lb)
    print('node is ', tuple(value(v) for v in model_sum_first_stg_vars))

    return output_lb, first_stg_nodes, [k_list, LB_list, add_node_history]


##################
##################
##################
##################                     set parameters here                   
##################
##################
##################
##################
csv_path = "data.csv"
max_scenarios = 5
n_per_axis = 5
N = max_scenarios
weights = (1.0, 0.01)
bounds = {
    "x": (-1e3, 1e3),
    "u": (None, None),
    "e": (-1e3, 1e3),
    "I": (-1e5, 1e5),
    "Kp": (0, 100),
    "Ki": (0, 100),
    "Kd": (0, 100),
}
solver = SolverFactory('gurobi')
solver.options.update({
    'MIPGap': 1e-2,
    'FeasibilityTol': 1e-6,
    'IntFeasTol':     1e-2,
    'OptimalityTol':  1e-6,
    'NumericFocus':   0,
    'Presolve':       2,
    'NonConvex':      2,
    'TimeLimit':      120,
})

model_list, first_stg_vars_list, m_tmpl_list, T = build_models_from_csv(
    csv_path, h=0.2, weights=weights, bounds=bounds,
    sp0=0.0, sp1=0.5, ku_col="tau_us", tau_col="tau_xs",
    disturb_prefix="disturbance_", setpoint_change_col="setpoint_change",
    max_scenarios=max_scenarios, skip=0
)

# persistent 求解器：每个场景一个——先 set_instance，再设参数
persistent_solvers = [GurobiPersistent() for _ in model_list]
for j, m in enumerate(model_list):
    opt = persistent_solvers[j]
    opt.set_instance(m)
    apply_params(opt)

# warm start 存储
global warm_solutions
warm_solutions = [None for _ in model_list]


# ---- 生成“轴向等距插值点”（）----
def build_full_grid_points(first_stg_vars, n_per_axis=10, endpoint=True, round_ndigits=12):
    """
    生成全维等距笛卡尔网格：
    - first_stg_vars: [Kp, Ki, Kd, ...]，每个变量需要有 lb/ub
    - n_per_axis: int 或 序列（逐维个数），例如 10 或 [10,12,8]
    - endpoint: 是否包含右端点（np.linspace 的参数）
    - round_ndigits: 为了稳定，把坐标四舍五入到固定小数位，避免浮点毛刺造成“相同点判不同”
    返回：list[tuple]，长度为 ∏ n_per_axis
    """
    lbs = [float(v.lb) for v in first_stg_vars]
    ubs = [float(v.ub) for v in first_stg_vars]
    d = len(first_stg_vars)

    if isinstance(n_per_axis, int):
        counts = [n_per_axis] * d
    else:
        if len(n_per_axis) != d:
            raise ValueError("n_per_axis 的长度必须等于维度数")
        counts = list(n_per_axis)

    # 每一维的等距坐标
    grids = [
        np.linspace(lbs[i], ubs[i], counts[i], endpoint=endpoint)
        for i in range(d)
    ]

    # 笛卡尔积 -> 点
    pts = [
        tuple(round(float(x), round_ndigits) for x in coords)
        for coords in it.product(*grids)
    ]
    return pts



first_stg_nodes = build_full_grid_points(first_stg_vars_list[0], n_per_axis=n_per_axis, endpoint=True, round_ndigits=12)
as_nodes_list = [[] for _ in range(N)]
for i in range(N):
    for node in first_stg_nodes:
        as_nodes_list[i].append(evaluate_Q_at(model_list[i], first_stg_vars_list[i], node, solver))

print(first_stg_nodes)
print('pts number is ',len(first_stg_nodes))


print('##################################################')
timeuse = []

for i in range(N):
    print('\nSolving scenario ', i)
    del_components(model_list[i])
    As, _ = add_nd_piecewise(
        model_list[i],
        first_stg_vars_list[i],
        first_stg_nodes,
        as_nodes_list[i],
        name=f"pw_scen_{i}",
        relation="=="
    )

    model_list[i].obj = Objective(expr=model_list[i].obj_expr - As, sense=minimize)

    # 先线性化，再绑定 persistent
    apply_pw_transform(model_list[i])
    assert_no_plf_left(model_list[i], where=f"scenario-{i}")
    apply_warm_values(model_list[i], warm_solutions[i])


    start = time.time()
    #results = opt.solve(tee=True)
    results = solver.solve(model_list[i], tee=True)
    end = time.time()


    print('**************************************************')
    print('**************************************************')
    print(f"scenario {i}, 计算Q与As最大差值（ms）用时: {end - start:.4f} 秒")
    print('**************************************************')
    print('**************************************************')
    timeuse.append(end - start)


print(len(first_stg_nodes),' nodes')
for i in range(N):
    print('\nSolving scenario ', i,'\n Time use is ', timeuse[i])

Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2689754
Academic license 2689754 - for non-commercial use only - registered to yi___@math.ubc.ca
Set parameter MIPGap to value 0.01
Set parameter IntFeasTol to value 1e-06
Set parameter NumericFocus to value 1
Set parameter Presolve to value 2
Set parameter NonConvex to value 2
Set parameter MIPGap to value 0.01
Set parameter IntFeasTol to value 1e-06
Set parameter NumericFocus to value 1
Set parameter Presolve to value 2
Set parameter NonConvex to value 2
Set parameter MIPGap to value 0.01
Set parameter IntFeasTol to value 1e-06
Set parameter NumericFocus to value 1
Set parameter Presolve to value 2
Set parameter NonConvex to value 2
Set parameter MIPGap to value 0.01
Set parameter IntFeasTol to value 1e-06
Set parameter NumericFocus to value 1
Set parameter Presolve to value 2
Set parameter NonConvex to value 2
Set parameter MIPGap to value 0.01
Set parameter IntFeasTol to value 1e-06
Set parameter Num

In [ ]:
# ---- 生成“轴向等距插值点”（总数 ~ 3 * n_per_axis）----
def build_axis_grid_points(first_stg_vars, n_per_axis=10):
    """
    对于 y=(Kp,Ki,Kd)，在每个轴上取 n_per_axis 个等距点，其余两维固定在中点。
    返回去重后的点列表（长度 ~ 3*n_per_axis）。
    """
    lbs = [float(v.lb) for v in first_stg_vars]
    ubs = [float(v.ub) for v in first_stg_vars]
    mids = [(a+b)/2.0 for a,b in zip(lbs, ubs)]

    pts = []
    for j in range(len(first_stg_vars)):
        grid = np.linspace(lbs[j], ubs[j], n_per_axis)
        for g in grid:
            y = list(mids)
            y[j] = float(g)
            pts.append(tuple(y))

    # 简单去重（容差 1e-9）
    out = []
    for p in pts:
        if not any(all(abs(a-b) <= 1e-9 for a,b in zip(p,q)) for q in out):
            out.append(p)
    return out

# ---- 生成随机点（均匀采样）----
def build_random_points(first_stg_vars, n_points=30, seed=17):
    rng = np.random.default_rng(seed)
    pts = []
    for _ in range(n_points):
        y = []
        for v in first_stg_vars:
            lb, ub = float(v.lb), float(v.ub)
            y.append(float(rng.uniform(lb, ub)))
        pts.append(tuple(y))
    return pts




# ---- 单场景基准：评估点→建 pw/As→解 "obj_expr - As" 并计时 ----
def benchmark_interp_on_points(model, first_stg_vars, points, solver, label="axis"):
    print(f"\n===== [{label}] | #points = {len(points)} =====")
    # 1) 逐点评估 Q_true（你的 evaluate_Q_at）
    values = []
    t0 = time.time()
    for pt in points:
        q = evaluate_Q_at(model, first_stg_vars, pt, solver)  # 解决第二阶段得“真值”
        values.append(q)
    t1 = time.time()
    print(f"eval Q on {len(points)} points: {t1 - t0:.3f}s")

    # 2) 用这些点建 piecewise：As == pw(y)
    #    （与现有流程一致：先删旧组件→add_nd_piecewise→transform）
    del_components(model)
    As, _ = add_nd_piecewise(
        model, first_stg_vars, points, values,
        name=f"grid_{label}", relation="=="
    )
    model.obj = pyo.Objective(expr=model.obj_expr - As, sense=pyo.minimize)

    apply_pw_transform(model)
    assert_no_plf_left(model, where=f"grid-{label}")

    # 3) 求解并计时
    t2 = time.time()
    results = solver.solve(model, tee=True)
    t3 = time.time()

    ok = (results.solver.status == SolverStatus.ok and
          results.solver.termination_condition == TerminationCondition.optimal)
    if not ok:
        print("⚠ 求解未最优：", results.solver.status, results.solver.termination_condition)

    # 统计一下规模（方便横向比较）
    n_vars = sum(v.size() for v in model.component_objects(pyo.Var, active=True))
    n_cons = sum(c.size() for c in model.component_objects(pyo.Constraint, active=True))
    n_bin = sum(v.size() for v in model.component_objects(pyo.Var, active=True)
                if hasattr(v, 'domain') and v.domain is pyo.Binary)

    print(f"transform+solve: {t3 - t2:.3f}s  |  Vars={n_vars}, Cons={n_cons}, ~Bin≈{n_bin}")
    # 返回总耗时
    return {"eval_time": t1 - t0, "solve_time": t3 - t2, "n_points": len(points),
            "n_vars": n_vars, "n_cons": n_cons, "n_bin": n_bin}

# ---- 一键跑两个方案：轴向等距 vs 随机点 ----
def quick_interp_benchmark(model_list, first_stg_vars_list, solver,
                           n_per_axis=10, n_random=30, scen_index=0):
    # 为了不污染你原始模型，克隆一份（保持完全同构）
    base_model, base_y = clone_and_get_vars(model_list[scen_index], first_stg_vars_list[scen_index])

    # 轴向等距（总点数 ~ 3*n_per_axis）
    axis_points = build_axis_grid_points(base_y, n_per_axis=n_per_axis)
    axis_res = benchmark_interp_on_points(base_model, base_y, axis_points, solver, label="axis")

    # 随机点（同量级）
    base_model2, base_y2 = clone_and_get_vars(model_list[scen_index], first_stg_vars_list[scen_index])
    rnd_points = build_random_points(base_y2, n_points=n_random, seed=17)
    rnd_res = benchmark_interp_on_points(base_model2, base_y2, rnd_points, solver, label="random")

    print("\n==== Summary ====")
    print(f"Axis  : n={axis_res['n_points']}, eval={axis_res['eval_time']:.3f}s, "
          f"solve={axis_res['solve_time']:.3f}s, Vars={axis_res['n_vars']}, "
          f"Cons={axis_res['n_cons']}, Bin≈{axis_res['n_bin']}")
    print(f"Random: n={rnd_res['n_points']}, eval={rnd_res['eval_time']:.3f}s, "
          f"solve={rnd_res['solve_time']:.3f}s, Vars={rnd_res['n_vars']}, "
          f"Cons={rnd_res['n_cons']}, Bin≈{rnd_res['n_bin']}")
    return axis_res, rnd_res